In [5]:
!nvidia-smi
!pip install -U ultralytics
import ultralytics
ultralytics.checks()


Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7108.1/8156.9 GB disk)


In [6]:
import requests, time, os, csv

out = "/kaggle/working/inat_images"
os.makedirs(out, exist_ok=True)

params = {
    "taxon_name": "Terrapene carolina",
    "quality_grade": "research",
    "photo_license": "cc0,cc-by,cc-by-nc",
    "per_page": 50,
}
target = 500
rows = []
page = 1

while len(rows) < target:
    params["page"] = page
    results = requests.get("https://api.inaturalist.org/v1/observations", params=params, timeout=30).json()["results"]
    if not results:
        break
    for obs in results:
        p = obs["photos"][0]
        url = p["url"].replace("square", "large")
        name = f"inat_{obs['id']}_{p['id']}.jpg"
        img = requests.get(url, timeout=30)
        if img.status_code == 200:
            open(f"{out}/{name}", "wb").write(img.content)
            rows.append([name, obs["id"], p["license_code"], obs["user"]["login"]])
        time.sleep(0.5)
    page += 1
    time.sleep(1)

with open("/kaggle/working/inat_credits.csv", "w", newline="") as f:
    csv.writer(f).writerows([["file", "observation_id", "license", "user"]] + rows)

print(len(rows), "images saved")

KeyboardInterrupt: 

In [ ]:
!cd /kaggle/working && zip -rq inat_images.zip inat_images inat_credits.csv
!ls -lh /kaggle/working/inat_images.zip

In [7]:
!ls /kaggle/working/inat_images | wc -l

500


In [1]:
!find /kaggle/input -maxdepth 4 -name "train"

/kaggle/input/datasets/moaaz1/turtle-labeled/train


In [2]:
import shutil, glob
shutil.copytree("/kaggle/input/datasets/moaaz1/turtle-labeled", "/kaggle/working/data")

open("/kaggle/working/turtle.yaml", "w").write("""path: /kaggle/working/data
train: train/images
val: valid/images
test: test/images
names:
  0: box_turtle
""")

f = glob.glob("/kaggle/working/data/train/labels/*.txt")[0]
print(open(f).read())

0 0.5771484375 0.3107822410147992 0.583984375 0.3192389006342495 0.5849609375 0.33403805496828753 0.6337890625 0.38266384778012685 0.6337890625 0.4143763213530655 0.6494140625 0.45665961945031713 0.7060546875 0.5285412262156448 0.724609375 0.5348837209302325 0.734375 0.5708245243128964 0.744140625 0.5623678646934461 0.744140625 0.5412262156448203 0.7373046875 0.5285412262156448 0.7353515625 0.5052854122621564 0.74609375 0.47780126849894294 0.7529296875 0.41014799154334036 0.748046875 0.35306553911205074 0.7412109375 0.3382663847780127 0.73046875 0.33403805496828753 0.7119140625 0.30655391120507397 0.6650390625 0.2854122621564482 0.6474609375 0.28964059196617337 0.6376953125 0.3361522198731501 0.61328125 0.3044397463002114 0.583984375 0.2959830866807611 0.5771484375 0.3107822410147992


In [3]:
import glob
for f in glob.glob("/kaggle/working/data/*/labels/*.txt"):
    out = []
    for line in open(f).read().strip().splitlines():
        p = line.split()
        if len(p) > 5:
            xs = [float(v) for v in p[1::2]]
            ys = [float(v) for v in p[2::2]]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            line = f"{p[0]} {(x0+x1)/2} {(y0+y1)/2} {x1-x0} {y1-y0}"
        out.append(line)
    open(f, "w").write("\n".join(out))

print(open(glob.glob("/kaggle/working/data/train/labels/*.txt")[0]).read())

0 0.6650390625 0.4281183932346723 0.17578125 0.2854122621564482


In [4]:
from ultralytics import YOLO
model = YOLO("yolo26s.pt")
model.train(data="/kaggle/working/turtle.yaml", epochs=60, imgsz=640,
            batch=16, project="/kaggle/working/runs", name="turtle1")

ModuleNotFoundError: No module named 'ultralytics'

In [5]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [6]:
from ultralytics import YOLO
model = YOLO("yolo26s.pt")
model.train(data="/kaggle/working/turtle.yaml", epochs=60, imgsz=640,
            batch=16, project="/kaggle/working/runs", name="turtle1")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/turtle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/60      5.77G      1.411      1.979    0.02221         45        640: 100% ━━━━━━━━━━━━ 22/22 4.9it/s 4.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 6.2it/s 0.6s0.3s
                   all        100        116      0.423      0.362       0.34       0.14

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/60      5.77G      1.581      1.698     0.0256         34        640: 100% ━━━━━━━━━━━━ 22/22 4.9it/s 4.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 6.0it/s 0.7s0.3s
                   all        100        116      0.624      0.543      0.538      0.249

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/60      5.77G       1.56      1.599    0.02389         32        640: 100% ━━━━━━━━━━━━ 22/22 4.9it/s 4.5s0.2s
                 Class     Images  Instances  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b91e4195a90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [8]:
!cd /kaggle/working && zip -rq turtle1.zip runs/turtle1

In [9]:
import requests, os, time
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/turtle1/weights/best.pt")
taxa = ["Trachemys scripta", "Chrysemys picta", "Chelydra serpentina",
        "Anaxyrus americanus", "Lithobates catesbeianus", "Agaricus", "Quercus"]

for taxon in taxa:
    folder = f"/kaggle/working/negatives/{taxon.replace(' ', '_')}"
    os.makedirs(folder, exist_ok=True)
    params = {"taxon_name": taxon, "quality_grade": "research",
              "photo_license": "cc0,cc-by,cc-by-nc", "per_page": 40}
    obs = requests.get("https://api.inaturalist.org/v1/observations", params=params, timeout=30).json()["results"]
    for o in obs:
        p = o["photos"][0]
        img = requests.get(p["url"].replace("square", "large"), timeout=30)
        if img.status_code == 200:
            open(f"{folder}/{o['id']}_{p['id']}.jpg", "wb").write(img.content)
        time.sleep(0.5)
    results = model.predict(folder, conf=0.25, verbose=False)
    hits = sum(1 for r in results if len(r.boxes) > 0)
    print(taxon, hits, "of", len(results), "images flagged")

Trachemys scripta 27 of 40 images flagged
Chrysemys picta 26 of 40 images flagged
Chelydra serpentina 31 of 40 images flagged
Anaxyrus americanus 25 of 40 images flagged
Lithobates catesbeianus 30 of 40 images flagged
Agaricus 11 of 40 images flagged
Quercus 7 of 40 images flagged


In [10]:
import glob
for conf in [0.25, 0.5, 0.7]:
    for folder in sorted(glob.glob("/kaggle/working/negatives/*")):
        results = model.predict(folder, conf=conf, verbose=False)
        hits = sum(1 for r in results if len(r.boxes) > 0)
        print(conf, folder.split("/")[-1], hits, "of", len(results))

0.25 Agaricus 11 of 40
0.25 Anaxyrus_americanus 25 of 40
0.25 Chelydra_serpentina 31 of 40
0.25 Chrysemys_picta 26 of 40
0.25 Lithobates_catesbeianus 30 of 40
0.25 Quercus 7 of 40
0.25 Trachemys_scripta 27 of 40
0.5 Agaricus 5 of 40
0.5 Anaxyrus_americanus 20 of 40
0.5 Chelydra_serpentina 23 of 40
0.5 Chrysemys_picta 20 of 40
0.5 Lithobates_catesbeianus 22 of 40
0.5 Quercus 4 of 40
0.5 Trachemys_scripta 23 of 40
0.7 Agaricus 1 of 40
0.7 Anaxyrus_americanus 15 of 40
0.7 Chelydra_serpentina 21 of 40
0.7 Chrysemys_picta 15 of 40
0.7 Lithobates_catesbeianus 14 of 40
0.7 Quercus 1 of 40
0.7 Trachemys_scripta 13 of 40


In [11]:
for conf in [0.25, 0.5, 0.7]:
    results = model.predict("/kaggle/working/data/valid/images", conf=conf, verbose=False)
    found = sum(1 for r in results if len(r.boxes) > 0)
    print(conf, "turtle found in", found, "of", len(results), "images")

0.25 turtle found in 96 of 100 images
0.5 turtle found in 93 of 100 images
0.7 turtle found in 91 of 100 images


In [12]:
import requests, os, time
more = ["Pseudacris crucifer", "Dryophytes versicolor", "Lithobates sylvaticus",
        "Plethodon cinereus", "Amanita"]
for taxon in more:
    folder = f"/kaggle/working/negatives/{taxon.replace(' ', '_')}"
    os.makedirs(folder, exist_ok=True)
    params = {"taxon_name": taxon, "quality_grade": "research",
              "photo_license": "cc0,cc-by,cc-by-nc", "per_page": 40}
    obs = requests.get("https://api.inaturalist.org/v1/observations", params=params, timeout=30).json()["results"]
    for o in obs:
        p = o["photos"][0]
        img = requests.get(p["url"].replace("square", "large"), timeout=30)
        if img.status_code == 200:
            open(f"{folder}/{o['id']}_{p['id']}.jpg", "wb").write(img.content)
        time.sleep(0.5)
    print(taxon, len(os.listdir(folder)))

Pseudacris crucifer 40
Dryophytes versicolor 40
Lithobates sylvaticus 40
Plethodon cinereus 40
Amanita 40


In [13]:
import glob, random, shutil, os
random.seed(0)
skip = ["Trachemys", "Chrysemys", "Chelydra"]
holdout = {}
for d in glob.glob("/kaggle/working/negatives/*"):
    if any(s in d for s in skip):
        continue
    imgs = sorted(glob.glob(d + "/*.jpg"))
    random.shuffle(imgs)
    n = len(imgs)
    parts = {"train": imgs[:int(n*0.6)], "valid": imgs[int(n*0.6):int(n*0.8)]}
    holdout[os.path.basename(d)] = imgs[int(n*0.8):]
    for split, files in parts.items():
        for f in files:
            name = "neg_" + os.path.basename(f)
            shutil.copy(f, f"/kaggle/working/data/{split}/images/{name}")
            open(f"/kaggle/working/data/{split}/labels/{name[:-4]}.txt", "w").close()
for c in glob.glob("/kaggle/working/data/*/labels.cache"):
    os.remove(c)
print({k: len(v) for k, v in holdout.items()})

{'Quercus': 8, 'Plethodon_cinereus': 8, 'Dryophytes_versicolor': 8, 'Anaxyrus_americanus': 8, 'Pseudacris_crucifer': 8, 'Agaricus': 8, 'Lithobates_catesbeianus': 8, 'Lithobates_sylvaticus': 8, 'Amanita': 8}


In [14]:
from ultralytics import YOLO
model2 = YOLO("yolo26s.pt")
model2.train(data="/kaggle/working/turtle.yaml", epochs=60, imgsz=640,
             batch=16, project="/kaggle/working/runs", name="turtle2")

Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/turtle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=turtle2, nbs=64, nms=None, opset=None, optimize=False, optimiz

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b92be987070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [15]:
model2 = YOLO("/kaggle/working/runs/turtle2/weights/best.pt")
for conf in [0.25, 0.5, 0.7]:
    for name, files in holdout.items():
        r = model2.predict(files, conf=conf, verbose=False)
        print(conf, name, sum(1 for x in r if len(x.boxes) > 0), "of", len(files), "flagged")
    r = model2.predict("/kaggle/working/data/test/images", conf=conf, verbose=False)
    print(conf, "TURTLES FOUND", sum(1 for x in r if len(x.boxes) > 0), "of", len(r))

0.25 Quercus 0 of 8 flagged
0.25 Plethodon_cinereus 0 of 8 flagged
0.25 Dryophytes_versicolor 0 of 8 flagged
0.25 Anaxyrus_americanus 1 of 8 flagged
0.25 Pseudacris_crucifer 1 of 8 flagged
0.25 Agaricus 0 of 8 flagged
0.25 Lithobates_catesbeianus 1 of 8 flagged
0.25 Lithobates_sylvaticus 0 of 8 flagged
0.25 Amanita 1 of 8 flagged
0.25 TURTLES FOUND 48 of 50
0.5 Quercus 0 of 8 flagged
0.5 Plethodon_cinereus 0 of 8 flagged
0.5 Dryophytes_versicolor 0 of 8 flagged
0.5 Anaxyrus_americanus 0 of 8 flagged
0.5 Pseudacris_crucifer 1 of 8 flagged
0.5 Agaricus 0 of 8 flagged
0.5 Lithobates_catesbeianus 0 of 8 flagged
0.5 Lithobates_sylvaticus 0 of 8 flagged
0.5 Amanita 0 of 8 flagged
0.5 TURTLES FOUND 48 of 50
0.7 Quercus 0 of 8 flagged
0.7 Plethodon_cinereus 0 of 8 flagged
0.7 Dryophytes_versicolor 0 of 8 flagged
0.7 Anaxyrus_americanus 0 of 8 flagged
0.7 Pseudacris_crucifer 0 of 8 flagged
0.7 Agaricus 0 of 8 flagged
0.7 Lithobates_catesbeianus 0 of 8 flagged
0.7 Lithobates_sylvaticus 0 of 8 fl

In [16]:
!cd /kaggle/working && zip -rq turtle2.zip runs/turtle2